In [ ]:
# CELL 1: Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print('Drive mounted.')

Mounted at /content/drive
Drive mounted.


In [ ]:
# CELL 2: Imports and constants
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import zipfile
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

DATA_ROOT     = '/content/avec2014/AVEC2014'
FRAME_ROOT    = '/content/avec2014_frames'
QR_CKPT       = '/content/drive/MyDrive/quantile_checkpoint_epoch7.pth'
OCC_CSV       = '/content/drive/MyDrive/avec2014_occlusion.csv'
GENDER_CSV    = '/content/drive/MyDrive/avec2014_gender.csv'
SAVE_DIR      = '/content/drive/MyDrive/avec2014_fuq_occlusion'

os.makedirs(SAVE_DIR, exist_ok=True)

CLIP_LEN    = 16
STRIDE      = 8
BATCH_SIZE  = 32
N_QUANTILES = 99
M_BINS      = 4
ALPHA       = 0.1
TARGET      = 1 - ALPHA

# Unzip if needed
marker = os.path.join(DATA_ROOT, 'labels.csv')
if not os.path.exists(marker):
    print('Extracting dataset...')
    with zipfile.ZipFile('/content/drive/MyDrive/AVEC2014.zip', 'r') as z:
        z.extractall('/content/avec2014')
    print('Done.')
else:
    print('Dataset already extracted.')

print(f'QR_CKPT exists: {os.path.exists(QR_CKPT)}')
print(f'OCC_CSV exists: {os.path.exists(OCC_CSV)}')

Device: cpu
Extracting dataset...
Done.
QR_CKPT exists: True
OCC_CSV exists: True


In [ ]:
# CELL 3: Load labels and occlusion annotations
def load_labels():
    df = pd.read_csv(os.path.join(DATA_ROOT, 'labels.csv'))
    labels = {}
    for _, row in df.iterrows():
        key = str(row['filename']).strip().replace('\\', '/')
        key = os.path.splitext(key)[0]
        labels[key] = float(row['BDI-II'])
    print(f'Labels loaded: {len(labels)}')
    return labels

labels  = load_labels()
occ_df  = pd.read_csv(OCC_CSV)
glasses_map = dict(zip(occ_df['filename'], occ_df['has_glasses']))
beard_map   = dict(zip(occ_df['filename'], occ_df['has_beard']))

print(f'Occlusion annotations: {len(occ_df)} videos')
print(f'Glasses distribution: {occ_df["has_glasses"].value_counts().to_dict()}')
print(f'Beard distribution:   {occ_df["has_beard"].value_counts().to_dict()}')

Labels loaded: 300
Occlusion annotations: 303 videos
Glasses distribution: {0: 265, 1: 38}
Beard distribution:   {0: 291, 1: 12}


In [ ]:
# CELL 4: Collect videos
def collect_videos(folders, labels):
    if isinstance(folders, str):
        folders = [folders]
    items = []
    for folder in folders:
        if not os.path.exists(folder):
            print(f'WARNING: not found: {folder}')
            continue
        for root, _, files in os.walk(folder):
            for f in sorted(files):
                if not f.lower().endswith('.mp4'):
                    continue
                path = os.path.join(root, f)
                rel  = os.path.relpath(path, DATA_ROOT).replace('\\', '/')
                stem = os.path.splitext(rel)[0]
                if stem in labels:
                    items.append((path, stem, labels[stem]))
    return items

TRAIN_DIRS = [
    os.path.join(DATA_ROOT, 'Training'),
    os.path.join(DATA_ROOT, 'Development'),
]
CAL_DIR  = os.path.join(DATA_ROOT, 'Testing', 'Northwind')
TEST_DIR = os.path.join(DATA_ROOT, 'Testing', 'Freeform')

train_items = collect_videos(TRAIN_DIRS, labels)
cal_items   = collect_videos(CAL_DIR,    labels)
test_items  = collect_videos(TEST_DIR,   labels)

print(f'Train: {len(train_items)}  Cal: {len(cal_items)}  Test: {len(test_items)}')

train_labels_arr = np.array([l for _, _, l in train_items])
LABEL_MEAN = float(train_labels_arr.mean())
LABEL_STD  = float(train_labels_arr.std())
print(f'Label norm — mean: {LABEL_MEAN:.2f}  std: {LABEL_STD:.2f}')

Train: 200  Cal: 50  Test: 50
Label norm — mean: 15.34  std: 12.07


In [ ]:
# CELL 5: Pre-extract frames (skips if done)
os.makedirs(FRAME_ROOT, exist_ok=True)

def extract_video(path, stem):
    out_dir = os.path.join(FRAME_ROOT, stem.replace('/', '_'))
    if os.path.exists(out_dir) and len(os.listdir(out_dir)) >= CLIP_LEN:
        return
    os.makedirs(out_dir, exist_ok=True)
    cap = cv2.VideoCapture(path)
    idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.resize(frame, (112, 112))
        cv2.imwrite(os.path.join(out_dir, f'{idx:05d}.jpg'), frame,
                    [cv2.IMWRITE_JPEG_QUALITY, 95])
        idx += 1
    cap.release()

all_items = train_items + cal_items + test_items
print(f'Extracting frames for {len(all_items)} videos...')
for path, stem, label in tqdm(all_items):
    extract_video(path, stem)
print('Extraction complete.')

Extracting frames for 300 videos...


100%|██████████| 300/300 [06:36<00:00,  1.32s/it]

Extraction complete.


In [ ]:
# CELL 6: Dataset classes
def load_frames_from_ssd(stem, start, n=CLIP_LEN):
    out_dir = os.path.join(FRAME_ROOT, stem.replace('/', '_'))
    frames  = []
    for i in range(start, start + n):
        fpath = os.path.join(out_dir, f'{i:05d}.jpg')
        if not os.path.exists(fpath):
            break
        frame = cv2.imread(fpath)
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)
    return frames

def count_frames(stem):
    out_dir = os.path.join(FRAME_ROOT, stem.replace('/', '_'))
    if not os.path.exists(out_dir):
        return 0
    return len([f for f in os.listdir(out_dir) if f.endswith('.jpg')])

def frames_to_tensor(frames):
    arr = np.stack(frames, axis=0).astype(np.float32) / 255.0
    arr = (arr - MEAN) / STD
    arr = arr.transpose(3, 0, 1, 2)
    return torch.from_numpy(arr)

def get_clip_starts(T, stride=STRIDE):
    return list(range(0, T - CLIP_LEN + 1, stride))

class AVEC2014Eval(Dataset):
    def __init__(self, items):
        self.samples = []
        for path, stem, label in items:
            T = count_frames(stem)
            if T >= CLIP_LEN:
                self.samples.append((stem, label))
        print(f'Eval videos: {len(self.samples)}')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        stem, label = self.samples[idx]
        T     = count_frames(stem)
        clips = []
        for s in get_clip_starts(T, stride=8):
            frames = load_frames_from_ssd(stem, s, CLIP_LEN)
            if len(frames) < CLIP_LEN:
                continue
            clips.append(frames_to_tensor(frames))
        if not clips:
            clips.append(torch.zeros(3, CLIP_LEN, 112, 112))
        return torch.stack(clips), torch.tensor(label, dtype=torch.float32), stem

cal_dataset  = AVEC2014Eval(cal_items)
test_dataset = AVEC2014Eval(test_items)

Eval videos: 50
Eval videos: 50


In [ ]:
# CELL 7: C3D Quantile model and load checkpoint
class C3DQuantile(nn.Module):
    def __init__(self, n_quantiles=99, dropout=0.5):
        super().__init__()
        self.conv1  = nn.Conv3d(3,   64,  kernel_size=(3,3,3), padding=(1,1,1))
        self.pool1  = nn.MaxPool3d(kernel_size=(1,2,2), stride=(1,2,2))
        self.conv2  = nn.Conv3d(64,  128, kernel_size=(3,3,3), padding=(1,1,1))
        self.pool2  = nn.MaxPool3d(kernel_size=(2,2,2), stride=(2,2,2))
        self.conv3a = nn.Conv3d(128, 256, kernel_size=(3,3,3), padding=(1,1,1))
        self.conv3b = nn.Conv3d(256, 256, kernel_size=(3,3,3), padding=(1,1,1))
        self.pool3  = nn.MaxPool3d(kernel_size=(2,2,2), stride=(2,2,2))
        self.conv4a = nn.Conv3d(256, 512, kernel_size=(3,3,3), padding=(1,1,1))
        self.conv4b = nn.Conv3d(512, 512, kernel_size=(3,3,3), padding=(1,1,1))
        self.pool4  = nn.MaxPool3d(kernel_size=(2,2,2), stride=(2,2,2))
        self.conv5a = nn.Conv3d(512, 512, kernel_size=(3,3,3), padding=(1,1,1))
        self.conv5b = nn.Conv3d(512, 512, kernel_size=(3,3,3), padding=(1,1,1))
        self.pool5  = nn.MaxPool3d(kernel_size=(2,2,2), stride=(2,2,2), padding=(0,1,1))
        self.relu    = nn.ReLU(inplace=True)
        self.fc6     = nn.Linear(8192, 4096)
        self.fc7     = nn.Linear(4096, 64)
        self.fc8     = nn.Linear(64, n_quantiles)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.relu(self.conv1(x));  x = self.pool1(x)
        x = self.relu(self.conv2(x));  x = self.pool2(x)
        x = self.relu(self.conv3a(x))
        x = self.relu(self.conv3b(x)); x = self.pool3(x)
        x = self.relu(self.conv4a(x))
        x = self.relu(self.conv4b(x)); x = self.pool4(x)
        x = self.relu(self.conv5a(x))
        x = self.relu(self.conv5b(x)); x = self.pool5(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(self.relu(self.fc6(x)))
        x = self.dropout(self.relu(self.fc7(x)))
        return self.fc8(x)

qr_model = C3DQuantile(n_quantiles=N_QUANTILES).to(device)
qr_model.load_state_dict(torch.load(QR_CKPT, map_location=device))
qr_model.eval()
print(f'Quantile model loaded from: {QR_CKPT}')

Quantile model loaded from: /content/drive/MyDrive/quantile_checkpoint_epoch7.pth


In [ ]:
# CELL 8 CPU-safe evaluation
QUANTILES = torch.linspace(0.01, 0.99, N_QUANTILES).to(device)
Q_LO_IDX  = int(ALPHA / 2 * N_QUANTILES)
Q_HI_IDX  = int((1 - ALPHA / 2) * N_QUANTILES) - 1

def get_video_quantiles_cpu(dataset, model):
    model.eval()
    results = []
    with torch.no_grad():
        for idx, (clips, label, stem) in enumerate(dataset):
            # Process one clip at a time on CPU
            all_preds = []
            for i in range(len(clips)):
                clip  = clips[i].unsqueeze(0)  # (1, 3, 16, 112, 112)
                pred  = model(clip)             # (1, 99)
                all_preds.append(pred.squeeze(0))
            q_norm = torch.stack(all_preds).mean(dim=0).numpy()
            q_pred = q_norm * LABEL_STD + LABEL_MEAN
            results.append((stem, label.item(), q_pred))
            if idx % 10 == 0:
                print(f'  {idx+1}/{len(dataset)} videos done...')
    return results

print('Evaluating calibration set...')
cal_results  = get_video_quantiles_cpu(cal_dataset,  qr_model)
print(f'Cal done: {len(cal_results)} videos')

print('Evaluating test set...')
test_results = get_video_quantiles_cpu(test_dataset, qr_model)
print(f'Test done: {len(test_results)} videos')

Evaluating calibration set...
  1/50 videos done...
  11/50 videos done...
  21/50 videos done...
  31/50 videos done...
  41/50 videos done...
Cal done: 50 videos
Evaluating test set...
  1/50 videos done...
  11/50 videos done...
  21/50 videos done...
  31/50 videos done...
  41/50 videos done...
Test done: 50 videos


In [ ]:
# CELL 9: Global CQR baseline
cal_scores = np.array([
    max(q[Q_LO_IDX] - y, y - q[Q_HI_IDX])
    for _, y, q in cal_results
])
beta = np.quantile(cal_scores, 1 - ALPHA)
print(f'Global conformal threshold beta = {beta:.4f}')

test_intervals_cqr = [
    (stem, y, q[Q_LO_IDX] - beta, q[Q_HI_IDX] + beta)
    for stem, y, q in test_results
]
picp_cqr = np.mean([lo <= y <= hi for _, y, lo, hi in test_intervals_cqr])
mpiw_cqr = np.mean([hi - lo for _, _, lo, hi in test_intervals_cqr])
print(f'CQR: PICP={picp_cqr:.4f}  MPIW={mpiw_cqr:.4f}')

Global conformal threshold beta = 14.9148
CQR: PICP=0.8800  MPIW=36.5307


In [ ]:
# CELL 10: FUQ core functions
# These are reusable for any sensitive attribute

def build_cal_df(cal_results, attr_map, attr_name):
    """Build calibration dataframe with occlusion attribute."""
    rows = []
    for stem, y_true, q_pred in cal_results:
        rows.append({
            'stem':   stem,
            'y_true': y_true,
            'y_lo':   q_pred[Q_LO_IDX],
            'y_hi':   q_pred[Q_HI_IDX],
            'r':      max(q_pred[Q_LO_IDX] - y_true, y_true - q_pred[Q_HI_IDX]),
            attr_name: attr_map.get(stem, 0)
        })
    return pd.DataFrame(rows)

def build_bins(cal_df, m_bins=M_BINS):
    """Create M equal-mass bins by depression severity."""
    cal_sorted = cal_df.sort_values('y_true').reset_index(drop=True)
    N_cal      = len(cal_sorted)
    bin_size   = N_cal // m_bins
    bins = []
    for m in range(m_bins):
        s  = m * bin_size
        e  = (m + 1) * bin_size if m < m_bins - 1 else N_cal
        bd = cal_sorted.iloc[s:e]
        bins.append({'m': m, 'l': bd['y_true'].min(), 'u': bd['y_true'].max(), 'data': bd})
    return bins

def avg_coverage(bins, r_hat, group_val, attr_name):
    covs = []
    for m, b in enumerate(bins):
        g = b['data'][b['data'][attr_name] == group_val]
        if len(g) == 0:
            continue
        r = r_hat[(m, group_val)]
        c = sum(1 for _, row in g.iterrows()
                if (row['y_lo']-r) <= row['y_true'] <= (row['y_hi']+r))
        covs.append(c / len(g))
    return np.mean(covs) if covs else 0.0

def slope_down(bin_data, r, group_val, attr_name):
    g = bin_data[bin_data[attr_name] == group_val]
    if len(g) == 0: return 0.0
    scores = sorted(g['r'].tolist())
    below  = [s for s in scores if s < r]
    if not below: return 0.0
    return (1.0/len(g)) / (r - below[-1] + 1e-8)

def slope_up(bin_data, r, group_val, attr_name):
    g = bin_data[bin_data[attr_name] == group_val]
    if len(g) == 0: return float('inf')
    scores = sorted(g['r'].tolist())
    above  = [s for s in scores if s > r]
    if not above: return float('inf')
    return (1.0/len(g)) / (above[0] - r + 1e-8)

def fuq_optimize(bins, r_hat, groups, attr_name, max_iter=500):
    """Fairness-aware optimization — paper Eqs. (6-11)."""
    for iteration in range(max_iter):
        cov = {g: avg_coverage(bins, r_hat, g, attr_name) for g in groups}
        if all(abs(cov[g] - TARGET) < 0.02 for g in groups):
            print(f'Converged at iteration {iteration}.')
            return r_hat, cov, iteration
        over  = max(groups, key=lambda g: cov[g])
        under = min(groups, key=lambda g: cov[g])
        if cov[over] <= TARGET:
            for g in groups:
                for m in range(M_BINS):
                    if slope_up(bins[m]['data'], r_hat[(m,g)], g, attr_name) < float('inf'):
                        r_hat[(m,g)] += 0.1
            continue
        sd_best = max(range(M_BINS), key=lambda m: slope_down(bins[m]['data'], r_hat[(m,over)],  over,  attr_name))
        su_best = min(range(M_BINS), key=lambda m: slope_up(bins[m]['data'],   r_hat[(m,under)], under, attr_name))
        gd = slope_down(bins[sd_best]['data'], r_hat[(sd_best,over)],  over,  attr_name)
        gu = slope_up(bins[su_best]['data'],   r_hat[(su_best,under)], under, attr_name)
        if gu > gd and cov[under] >= TARGET - 0.02:
            print(f'Slope condition met at iteration {iteration}.')
            return r_hat, cov, iteration
        r_hat[(sd_best,over)]  -= 0.1
        r_hat[(su_best,under)] += 0.1
        if iteration % 50 == 0:
            print(f'Iter {iteration:4d} | {groups[0]}: {cov[groups[0]]:.4f}  {groups[1]}: {cov[groups[1]]:.4f}')
    print('Max iterations reached.')
    return r_hat, cov, max_iter

def fuq_intervals(test_results, bins, r_hat, attr_map, attr_name):
    """Apply FUQ intervals to test set."""
    rows = []
    for stem, y_true, q_pred in test_results:
        y_lo = q_pred[Q_LO_IDX]
        y_hi = q_pred[Q_HI_IDX]
        g    = attr_map.get(stem, 0)
        ulo, uhi = float('inf'), float('-inf')
        for m, b in enumerate(bins):
            r      = r_hat[(m, g)]
            lo     = max(y_lo-r, b['l'])
            hi     = min(y_hi+r, b['u'])
            if lo <= hi:
                ulo = min(ulo, lo)
                uhi = max(uhi, hi)
        if ulo == float('inf'):
            r   = r_hat[(0, g)]
            ulo = y_lo - r
            uhi = y_hi + r
        rows.append({
            'stem':     stem,
            'y_true':   y_true,
            'y_pred':   q_pred[49],
            'lo':       ulo,
            'hi':       uhi,
            attr_name:  g,
            'covered':  ulo <= y_true <= uhi
        })
    return pd.DataFrame(rows)

def print_results(df, groups, group_labels, attr_name, method_name):
    print(f'\n=== {method_name} Results ===')
    print(f'Overall PICP: {df["covered"].mean():.4f}  MPIW: {(df["hi"]-df["lo"]).mean():.4f}')
    picps = {}
    for g, lbl in zip(groups, group_labels):
        grp  = df[df[attr_name]==g]
        picp = grp['covered'].mean() if len(grp) > 0 else float('nan')
        mpiw = (grp['hi']-grp['lo']).mean() if len(grp) > 0 else float('nan')
        mae  = (grp['y_true']-grp['y_pred']).abs().mean() if len(grp) > 0 else float('nan')
        picps[g] = picp
        print(f'  {lbl:15s} N={len(grp):3d}  PICP={picp:.4f}  MPIW={mpiw:.4f}  MAE={mae:.4f}')
    gap = abs(picps[groups[0]] - picps[groups[1]])
    print(f'  PICP Gap: {gap:.4f}')
    return picps, gap

print('FUQ functions defined.')

FUQ functions defined.


In [ ]:
# CELL 11: FUQ with GLASSES as sensitive attribute
# Groups: 0=No Glasses, 1=Glasses

print('=' * 50)
print('FUQ with GLASSES as sensitive attribute')
print('=' * 50)

GLASSES_GROUPS = [0, 1]
GLASSES_LABELS = ['No Glasses', 'Glasses']
ATTR           = 'has_glasses'

# Build calibration data
cal_df_glasses = build_cal_df(cal_results, glasses_map, ATTR)
bins_glasses   = build_bins(cal_df_glasses)

print(f'\nCalibration set glasses distribution:')
print(cal_df_glasses[ATTR].value_counts().rename({0:'No Glasses', 1:'Glasses'}))

for m, b in enumerate(bins_glasses):
    ng = len(b['data'][b['data'][ATTR]==0])
    g  = len(b['data'][b['data'][ATTR]==1])
    print(f'Bin {m+1}: [{b["l"]:.1f}, {b["u"]:.1f}]  No Glasses={ng}  Glasses={g}')

# Initialise thresholds with global beta
r_hat_glasses = {(m, g): beta for m in range(M_BINS) for g in GLASSES_GROUPS}

print(f'\nInitial coverage:')
for g, lbl in zip(GLASSES_GROUPS, GLASSES_LABELS):
    cov = avg_coverage(bins_glasses, r_hat_glasses, g, ATTR)
    print(f'  {lbl}: {cov:.4f}')

# Run fairness optimization
print('\n=== Fairness-Aware Optimization ===')
r_hat_glasses, final_cov_glasses, n_iter = fuq_optimize(
    bins_glasses, r_hat_glasses, GLASSES_GROUPS, ATTR
)

print(f'\nFinal calibration coverage:')
for g, lbl in zip(GLASSES_GROUPS, GLASSES_LABELS):
    print(f'  {lbl}: {final_cov_glasses[g]:.4f}  (target {TARGET:.2f})')
print(f'  Gap: {abs(final_cov_glasses[0]-final_cov_glasses[1]):.4f}')

# Apply to test set
fuq_glasses_df = fuq_intervals(test_results, bins_glasses, r_hat_glasses, glasses_map, ATTR)

# CQR baseline for glasses
cqr_glasses_df = pd.DataFrame([
    {'stem': stem, 'y_true': y, 'lo': lo, 'hi': hi,
     ATTR: glasses_map.get(stem, 0), 'covered': lo<=y<=hi,
     'y_pred': q[49]}
    for (stem, y, lo, hi), (_, _, q) in zip(test_intervals_cqr, test_results)
])

print('\n--- CQR (glasses) ---')
cqr_picps_g, cqr_gap_g = print_results(cqr_glasses_df, GLASSES_GROUPS, GLASSES_LABELS, ATTR, 'CQR')

print('\n--- FUQ (glasses as sensitive attribute) ---')
fuq_picps_g, fuq_gap_g = print_results(fuq_glasses_df, GLASSES_GROUPS, GLASSES_LABELS, ATTR, 'FUQ')

print(f'\nGap reduction: {cqr_gap_g:.4f} → {fuq_gap_g:.4f}')
print(f'MPIW change:   CQR={cqr_glasses_df["hi"].sub(cqr_glasses_df["lo"]).mean():.4f}  FUQ={fuq_glasses_df["hi"].sub(fuq_glasses_df["lo"]).mean():.4f}')

FUQ with GLASSES as sensitive attribute

Calibration set glasses distribution:
has_glasses
No Glasses    42
Glasses        8
Name: count, dtype: int64
Bin 1: [0.0, 3.0]  No Glasses=11  Glasses=1
Bin 2: [3.0, 12.0]  No Glasses=10  Glasses=2
Bin 3: [12.0, 21.0]  No Glasses=10  Glasses=2
Bin 4: [22.0, 43.0]  No Glasses=11  Glasses=3

Initial coverage:
  No Glasses: 0.9091
  Glasses: 0.7500

=== Fairness-Aware Optimization ===
Iter    0 | 0: 0.9091  1: 0.7500
Iter   50 | 0: 0.9091  1: 0.7500
Iter  100 | 0: 0.9091  1: 0.7500
Iter  150 | 0: 0.9091  1: 0.7500
Iter  200 | 0: 0.9091  1: 0.7500
Iter  250 | 0: 0.9091  1: 0.7500
Iter  300 | 0: 0.9091  1: 0.7500
Iter  350 | 0: 0.9091  1: 0.7500
Iter  400 | 0: 0.9091  1: 0.7500
Iter  450 | 0: 0.9091  1: 0.7500
Max iterations reached.

Final calibration coverage:
  No Glasses: 0.9091  (target 0.90)
  Glasses: 1.0000  (target 0.90)
  Gap: 0.0909

--- CQR (glasses) ---

=== CQR Results ===
Overall PICP: 0.8800  MPIW: 36.5307
  No Glasses      N= 42  PI

In [ ]:
# CELL 12: FUQ with BEARD as sensitive attribute
# NOTE: Only 2 beard samples in calibration set
# Results are directionally informative but not statistically reliable

print('=' * 50)
print('FUQ with BEARD as sensitive attribute')
print('NOTE: Only 2 beard samples in calibration set')
print('Results should be interpreted with caution')
print('=' * 50)

BEARD_GROUPS = [0, 1]
BEARD_LABELS = ['No Beard', 'Beard']
ATTR_B       = 'has_beard'

cal_df_beard = build_cal_df(cal_results, beard_map, ATTR_B)
bins_beard   = build_bins(cal_df_beard)

print(f'\nCalibration set beard distribution:')
print(cal_df_beard[ATTR_B].value_counts().rename({0:'No Beard', 1:'Beard'}))

for m, b in enumerate(bins_beard):
    nb = len(b['data'][b['data'][ATTR_B]==0])
    bd = len(b['data'][b['data'][ATTR_B]==1])
    print(f'Bin {m+1}: [{b["l"]:.1f}, {b["u"]:.1f}]  No Beard={nb}  Beard={bd}')

r_hat_beard = {(m, g): beta for m in range(M_BINS) for g in BEARD_GROUPS}

print(f'\nInitial coverage:')
for g, lbl in zip(BEARD_GROUPS, BEARD_LABELS):
    cov = avg_coverage(bins_beard, r_hat_beard, g, ATTR_B)
    print(f'  {lbl}: {cov:.4f}')

print('\n=== Fairness-Aware Optimization ===')
r_hat_beard, final_cov_beard, n_iter_b = fuq_optimize(
    bins_beard, r_hat_beard, BEARD_GROUPS, ATTR_B
)

print(f'\nFinal calibration coverage:')
for g, lbl in zip(BEARD_GROUPS, BEARD_LABELS):
    print(f'  {lbl}: {final_cov_beard[g]:.4f}  (target {TARGET:.2f})')
print(f'  Gap: {abs(final_cov_beard[0]-final_cov_beard[1]):.4f}')

fuq_beard_df = fuq_intervals(test_results, bins_beard, r_hat_beard, beard_map, ATTR_B)

cqr_beard_df = pd.DataFrame([
    {'stem': stem, 'y_true': y, 'lo': lo, 'hi': hi,
     ATTR_B: beard_map.get(stem, 0), 'covered': lo<=y<=hi,
     'y_pred': q[49]}
    for (stem, y, lo, hi), (_, _, q) in zip(test_intervals_cqr, test_results)
])

print('\n--- CQR (beard) ---')
cqr_picps_b, cqr_gap_b = print_results(cqr_beard_df, BEARD_GROUPS, BEARD_LABELS, ATTR_B, 'CQR')

print('\n--- FUQ (beard as sensitive attribute) ---')
fuq_picps_b, fuq_gap_b = print_results(fuq_beard_df, BEARD_GROUPS, BEARD_LABELS, ATTR_B, 'FUQ')

print(f'\nGap reduction: {cqr_gap_b:.4f} → {fuq_gap_b:.4f}')
print(f'MPIW change:   CQR={cqr_beard_df["hi"].sub(cqr_beard_df["lo"]).mean():.4f}  FUQ={fuq_beard_df["hi"].sub(fuq_beard_df["lo"]).mean():.4f}')

FUQ with BEARD as sensitive attribute
NOTE: Only 2 beard samples in calibration set
Results should be interpreted with caution

Calibration set beard distribution:
has_beard
No Beard    48
Beard        2
Name: count, dtype: int64
Bin 1: [0.0, 3.0]  No Beard=11  Beard=1
Bin 2: [3.0, 12.0]  No Beard=12  Beard=0
Bin 3: [12.0, 21.0]  No Beard=11  Beard=1
Bin 4: [22.0, 43.0]  No Beard=14  Beard=0

Initial coverage:
  No Beard: 0.9237
  Beard: 0.5000

=== Fairness-Aware Optimization ===
Iter    0 | 0: 0.9237  1: 0.5000
Iter   50 | 0: 0.9058  1: 1.0000
Iter  100 | 0: 0.9058  1: 1.0000
Iter  150 | 0: 0.9058  1: 1.0000
Iter  200 | 0: 0.9058  1: 1.0000
Iter  250 | 0: 0.9058  1: 1.0000
Iter  300 | 0: 0.9058  1: 1.0000
Iter  350 | 0: 0.9058  1: 1.0000
Iter  400 | 0: 0.9058  1: 1.0000
Iter  450 | 0: 0.9058  1: 1.0000
Max iterations reached.

Final calibration coverage:
  No Beard: 0.9058  (target 0.90)
  Beard: 0.5000  (target 0.90)
  Gap: 0.4058

--- CQR (beard) ---

=== CQR Results ===
Overall PI

In [ ]:
# CELL 13: Full comparison table
# CQR vs FUQ(gender) vs FUQ(glasses) vs FUQ(beard)

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

print('=' * 70)
print('FULL COMPARISON: CQR vs FUQ with different sensitive attributes')
print('=' * 70)
print(f'{"Method":30s} {"PICP":>8s} {"MPIW":>8s} {"Group 0 PICP":>14s} {"Group 1 PICP":>14s} {"Gap":>8s}')
print('-' * 84)

rows = [
    ('CQR (baseline)',          cqr_glasses_df, GLASSES_GROUPS, GLASSES_LABELS, ATTR),
    ('FUQ (glasses attr)',      fuq_glasses_df, GLASSES_GROUPS, GLASSES_LABELS, ATTR),
    ('FUQ (beard attr)*',       fuq_beard_df,   BEARD_GROUPS,   BEARD_LABELS,   ATTR_B),
]

for name, df, groups, labels_, attr in rows:
    picp_all = df['covered'].mean()
    mpiw_all = (df['hi'] - df['lo']).mean()
    p0 = df[df[attr]==groups[0]]['covered'].mean()
    p1 = df[df[attr]==groups[1]]['covered'].mean() if (df[attr]==groups[1]).any() else float('nan')
    gap = abs(p0 - p1)
    print(f'{name:30s} {picp_all:8.4f} {mpiw_all:8.4f} {p0:14.4f} {p1:14.4f} {gap:8.4f}')

print('\n* Beard results based on N=2 calibration samples — interpret with caution')

FULL COMPARISON: CQR vs FUQ with different sensitive attributes
Method                             PICP     MPIW   Group 0 PICP   Group 1 PICP      Gap
------------------------------------------------------------------------------------
CQR (baseline)                   0.8800  36.5307         0.9048         0.7500   0.1548
FUQ (glasses attr)               0.8800  30.5726         0.9048         0.7500   0.1548
FUQ (beard attr)*                0.8400  28.4635         0.8542         0.5000   0.3542

* Beard results based on N=2 calibration samples — interpret with caution


In [ ]:
# CELL 14: Save results
fuq_glasses_df.to_csv(os.path.join(SAVE_DIR, 'fuq_glasses_results.csv'), index=False)
fuq_beard_df.to_csv(os.path.join(SAVE_DIR, 'fuq_beard_results.csv'), index=False)
print(f'Results saved to: {SAVE_DIR}')

# Summary
print('\n===== Final Summary =====')
print(f'Glasses — CQR Gap: {cqr_gap_g:.4f}  FUQ Gap: {fuq_gap_g:.4f}  Reduction: {cqr_gap_g-fuq_gap_g:.4f}')
print(f'Beard   — CQR Gap: {cqr_gap_b:.4f}  FUQ Gap: {fuq_gap_b:.4f}  Reduction: {cqr_gap_b-fuq_gap_b:.4f}')
print(f'CQR MPIW:          {cqr_glasses_df["hi"].sub(cqr_glasses_df["lo"]).mean():.4f}')
print(f'FUQ (glasses) MPIW:{fuq_glasses_df["hi"].sub(fuq_glasses_df["lo"]).mean():.4f}')
print(f'FUQ (beard)   MPIW:{fuq_beard_df["hi"].sub(fuq_beard_df["lo"]).mean():.4f}')

Results saved to: /content/drive/MyDrive/avec2014_fuq_occlusion

===== Final Summary =====
Glasses — CQR Gap: 0.1548  FUQ Gap: 0.1548  Reduction: 0.0000
Beard   — CQR Gap: 0.3958  FUQ Gap: 0.3542  Reduction: 0.0417
CQR MPIW:          36.5307
FUQ (glasses) MPIW:30.5726
FUQ (beard)   MPIW:28.4635
